In [2]:
import cv2
import math
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

previous_position = {}
previous_frame = {}

cap = cv2.VideoCapture("car_video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)

PIXELS_PER_METER = 10
frame_no = 0

# Output video
# Video size
width = 900
height = 600

# AVI output
fourcc = cv2.VideoWriter_fourcc(*"XVID")

out = cv2.VideoWriter(
    "car_speed_detection.avi",
    fourcc,
    fps,
    (width, height)
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_no += 1

    frame = cv2.resize(frame, (900, 600))

    result = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[2],
        verbose=False
    )[0]

    output = result.plot()

    if result.boxes.id is not None:

        ids = result.boxes.id.int().cpu().tolist()
        boxes = result.boxes.xyxy.cpu().numpy()

        for box, track_id in zip(boxes, ids):

            x1, y1, x2, y2 = map(int, box)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            speed = 0

            if track_id in previous_position:

                old_x, old_y = previous_position[track_id]
                old_frame = previous_frame[track_id]

                distance = math.hypot(cx - old_x, cy - old_y)
                time = (frame_no - old_frame) / fps

                pixel_speed = distance / time
                speed = (pixel_speed / PIXELS_PER_METER) * 3.6

                cv2.line(
                    output,
                    (old_x, old_y),
                    (cx, cy),
                    (255, 0, 0),
                    2
                )

            cv2.putText(
                output,
                f"{speed:.1f} km/h",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

            previous_position[track_id] = (cx, cy)
            previous_frame[track_id] = frame_no

    cv2.imshow("Car Speed", output)
    out.write(output)

    if cv2.waitKey(1) & 0xFF == ord(" "):
        break

cap.release()
cv2.destroyAllWindows()